# Lab3- Cyber Risk Regression and Logistic classification (Student Template)

⚠️**IMPORTANT — READ FIRST**

Do NOT edit this template directly

1. Click File → Save a copy in Drive
2. Rename your notebook: `Lab3_YourName.ipynb`.
3. Complete all tasks by writing code in the empty cells.
4. Do NOT modify cells marked **PROVIDED — DO NOT MODIFY**.
5. Run all cells before submission so outputs are visible.
6. Download and submit the `.ipynb` file before the deadline.






















**Learning Objectives**

By the end of this lab, you will be able to:

1-Build multiple and polynomial regression models

2-Evaluate regression using MAE, RMSE, R²

3-Apply feature scaling

4-Use Ridge and Lasso regularization

5-Apply cross-validation

6-Convert a regression problem into binary classification

7-Train and evaluate Logistic Regression

8-Interpret results in a cyber risk context

**Dataset Description**

Each row represents one network session.

**Features (X)**

failed_logins: number of failed login attempts

data_volume_mb: data transferred during session (MB)

unusual_time: 1 if login occurred at unusual time, else 0

patch_age_days: days since last system patch

admin_login: 1 if admin account used

**Target (y)**

risk_score: cyber risk score between 0 and 100

This dataset is synthetic, created only for learning purposes.

**Imports** (PROVIDED — DO NOT MODIFY)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


**Dataset Generation** (PROVIDED — DO NOT MODIFY)

In [ ]:
# ======================================================
# Synthetic Cyber Risk Dataset (PROVIDED — DO NOT MODIFY)
# ======================================================
# Each row represents one network session.
# This dataset is generated only for learning purposes.
# The goal is to predict a cyber risk score (0–100).

import numpy as np
import pandas as pd

np.random.seed(42)
n_samples = 500

# -------------------------------
# Input Features (X)
# -------------------------------

# Number of failed login attempts in the session
failed_logins = np.random.poisson(lam=2, size=n_samples)

# Amount of data transferred during the session (in MB)
data_volume_mb = np.random.normal(loc=300, scale=120, size=n_samples).clip(50, 1000)

# Whether the session occurred at an unusual time (1 = yes, 0 = no)
unusual_time = np.random.binomial(1, 0.25, size=n_samples)

# Days since the system was last patched
patch_age_days = np.random.randint(0, 365, size=n_samples)

# Whether an admin account was used in the session
admin_login = np.random.binomial(1, 0.15, size=n_samples)

# -------------------------------
# Target Variable (y)
# -------------------------------
# Cyber risk score:
# - Higher values indicate higher security risk
# - Combines linear, nonlinear, and noisy effects
risk_score = (
    5 * failed_logins +
    0.04 * data_volume_mb +
    15 * unusual_time +
    0.03 * patch_age_days +
    20 * admin_login +
    0.5 * failed_logins**2 +           # nonlinear effect
    np.random.normal(0, 5, size=n_samples)  # noise
)

# Limit risk score to the range 0–100
risk_score = np.clip(risk_score, 0, 100)

# -------------------------------
# Create Final Dataset
# -------------------------------
df = pd.DataFrame({
    "failed_logins": failed_logins,
    "data_volume_mb": data_volume_mb,
    "unusual_time": unusual_time,
    "patch_age_days": patch_age_days,
    "admin_login": admin_login,
    "risk_score": risk_score
})

# Preview the dataset
df.head()



,failed_logins,data_volume_mb,unusual_time,patch_age_days,admin_login,risk_score
0,4,491.340608,0,216,0,54.978987
1,1,198.364638,0,119,0,15.001039
2,3,181.032918,0,174,0,21.995223
3,3,50.000000,0,22,0,11.908258
4,1,223.324590,0,212,1,43.741031


**Train–Test Split** (PROVIDED — DO NOT MODIFY)

In [ ]:
# -------------------------------------------
# Train–Test Split-(PROVIDED — DO NOT MODIFY)
# -------------------------------------------
# X: input features, y: target (risk score)
X = df.drop(columns=["risk_score"])
y = df["risk_score"]

# Split data into training and test sets
# Test set simulates unseen (future) network sessions
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

**Regression Metrics Helper**(PROVIDED — DO NOT MODIFY)

In [ ]:
# -------------------------------------------------------
# Regression Evaluation Metrics-(PROVIDED — DO NOT MODIFY)
# -------------------------------------------------------
# This function reports common regression metrics:
# MAE  -> average prediction error
# RMSE -> penalizes large errors
# R²   -> proportion of variance explained
def regression_report(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(model_name)
    print(f"  MAE : {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  R²  : {r2:.3f}")


# ✅ Student Tasks

Write your code in the empty code cells under each task.

**Tip:** call Regression Evaluation Metrics using `regression_report(y_test, y_pred, "Model Name")`.

**Task 1 — Multiple Linear Regression**

**Goal:**  
Establish a baseline model for predicting cyber risk score.

**What to do:**  
- Train a Multiple Linear Regression model on the training data.
- Evaluate it on the test set.

**What to observe:**  
- Record MAE, RMSE, and R².
- This baseline will be used to compare all other regression models.

In [ ]:
# Student code here

from sklearn.linear_model import LinearRegression

# Train baseline model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Evaluate
regression_report(y_test, y_pred_lr, "Multiple Linear Regression (Baseline)")

# Feature importance analysis
print("\n" + "="*50)
print("FEATURE COEFFICIENTS")
print("="*50)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_,
    'Absolute Impact': np.abs(lr_model.coef_)
}).sort_values('Absolute Impact', ascending=False)
print(feature_importance.to_string(index=False))
print(f"\nIntercept: {lr_model.intercept_:.4f}")

Multiple Linear Regression (Baseline)
  MAE : 4.307
  RMSE: 5.388
  R²  : 0.892

FEATURE COEFFICIENTS
       Feature  Coefficient  Absolute Impact
   admin_login    20.866194        20.866194
  unusual_time    14.317582        14.317582
 failed_logins     7.387997         7.387997
data_volume_mb     0.040344         0.040344
patch_age_days     0.027571         0.027571

Intercept: -1.5314


### Short Info for Task 2 -Why do we transform the input features?

Linear Regression can only learn **straight-line (linear) relationships** between the input features and the target.

However, in cyber risk assessment, relationships are often **nonlinear**.
For example:
- A few failed logins may have little impact on risk,
- But many failed logins can increase risk much faster.

Polynomial feature transformation creates **new features** (such as squared terms) from the original inputs.
This allows a linear regression model to learn **curved (nonlinear) patterns**.

Important notes:
- We transform **only the input features (X)**, never the target (y).
- We use `fit_transform` on the training data and `transform` on the test data
  to ensure the test data does not influence the training process.

In short:
> Polynomial regression is still linear regression, but applied to transformed features.


## Task 2 — Polynomial Regression (Degree = 2)

**Goal:**  
Capture nonlinear relationships in the cyber risk data.

**What to do:**  
- Use polynomial features of degree 2.
- Train a regression model using these features.

**What to observe:**  
- Compare performance with linear regression.
- Check whether the model improves or begins to overfit.




In [ ]:
# Student code here

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Create pipeline with polynomial features
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)

# Train and predict
poly_model.fit(X_train, y_train)
y_pred_poly = poly_model.predict(X_test)

# Evaluate
regression_report(y_test, y_pred_poly, "\nPolynomial Regression (degree=2)")

# Compare with baseline
r2_improvement = r2_score(y_test, y_pred_poly) - r2_score(y_test, y_pred_lr)
rmse_improvement = np.sqrt(mean_squared_error(y_test, y_pred_lr)) - np.sqrt(mean_squared_error(y_test, y_pred_poly))
print(f"\nImprovement over baseline:")
print(f"  R² change: {r2_improvement:+.4f}")
print(f"  RMSE change: {rmse_improvement:+.4f}")


Polynomial Regression (degree=2)
  MAE : 4.383
  RMSE: 5.613
  R²  : 0.883

Improvement over baseline:
  R² change: -0.0092
  RMSE change: -0.2249


## Task 3 — Ridge Regression (L2 Regularization)

**Goal:**  
Reduce overfitting by penalizing large coefficients.Ridge regression adds a regularization term that makes the model
more stable without changing the number of features.

**What to do:**  
- Apply feature scaling.
- Train Ridge Regression using 5-fold cross-validation.

**What to observe:**  
- Observe how regularization affects RMSE.
- Compare stability with previous models.


In [ ]:
# Student code here

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold

# Create pipeline with scaling and ridge
ridge_pipe = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0, random_state=42)
)

# Perform 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_ridge = cross_val_score(
    ridge_pipe, X_train, y_train,
    cv=cv, scoring='neg_root_mean_squared_error'
)

print("\n" + "="*50)
print("RIDGE REGRESSION CV RESULTS")
print("="*50)
print(f"CV RMSE scores: {[-score for score in cv_scores_ridge]}")
print(f"CV RMSE - Mean: {-cv_scores_ridge.mean():.4f}, Std: {cv_scores_ridge.std():.4f}")

# Train on full training set
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_test)

# Evaluate on test set
regression_report(y_test, y_pred_ridge, "\nRidge Regression (Test Set)")

# Try different alpha values to find optimal
print("\n" + "="*50)
print("HYPERPARAMETER TUNING")
print("="*50)
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_score = float('inf')

for alpha in alphas:
    ridge_test = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha, random_state=42)
    )
    cv_scores = cross_val_score(
        ridge_test, X_train, y_train,
        cv=cv, scoring='neg_root_mean_squared_error'
    )
    mean_rmse = -cv_scores.mean()
    print(f"Alpha={alpha:6.2f}: CV RMSE = {mean_rmse:.4f}")

    if mean_rmse < best_score:
        best_score = mean_rmse
        best_alpha = alpha

print(f"\nBest alpha: {best_alpha} (CV RMSE: {best_score:.4f})")


RIDGE REGRESSION CV RESULTS
CV RMSE scores: [np.float64(5.18132534744815), np.float64(4.557651235842766), np.float64(5.497129066701578), np.float64(4.21898377383279), np.float64(4.800025104659894)]
CV RMSE - Mean: 4.8510, Std: 0.4505

Ridge Regression (Test Set)
  MAE : 4.308
  RMSE: 5.386
  R²  : 0.892

HYPERPARAMETER TUNING
Alpha=  0.01: CV RMSE = 4.8520
Alpha=  0.10: CV RMSE = 4.8519
Alpha=  1.00: CV RMSE = 4.8510
Alpha= 10.00: CV RMSE = 4.8621
Alpha=100.00: CV RMSE = 5.9985

Best alpha: 1.0 (CV RMSE: 4.8510)


## Task 4 — Lasso Regression (L1 Regularization)

**Goal:**  
Perform regularization and implicit feature selection.
Lasso can shrink some coefficients to zero, effectively removing features.

**What to do:**  
- Apply Lasso Regression with feature scaling.
- Use cross-validation to evaluate performance.
Note: After cross-validation, we fit Lasso once on the full training set to inspect coefficients.


**What to observe:**  
- Identify coefficients that are reduced to zero.If none become 0, try a larger alpha and observe the change
- Compare performance with Ridge Regression.


In [ ]:
# Student code here

from sklearn.linear_model import Lasso

# Create pipeline with scaling and lasso
lasso_pipe = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1, max_iter=10000, random_state=42)
)

# Perform 5-fold cross-validation
cv_scores_lasso = cross_val_score(
    lasso_pipe, X_train, y_train,
    cv=cv, scoring='neg_root_mean_squared_error'
)

print("\n" + "="*50)
print("LASSO REGRESSION CV RESULTS")
print("="*50)
print(f"CV RMSE scores: {[-score for score in cv_scores_lasso]}")
print(f"CV RMSE - Mean: {-cv_scores_lasso.mean():.4f}, Std: {cv_scores_lasso.std():.4f}")

# Train on full training set
lasso_pipe.fit(X_train, y_train)
y_pred_lasso = lasso_pipe.predict(X_test)

# Evaluate on test set
regression_report(y_test, y_pred_lasso, "\nLasso Regression (Test Set)")

# Try different alpha values to see feature selection
print("\n" + "="*50)
print("LASSO FEATURE SELECTION ANALYSIS")
print("="*50)
alphas_lasso = [0.001, 0.01, 0.1, 1.0, 10.0]

for alpha in alphas_lasso:
    lasso_test = make_pipeline(
        StandardScaler(),
        Lasso(alpha=alpha, max_iter=10000, random_state=42)
    )
    lasso_test.fit(X_train, y_train)

    # Get coefficients
    lasso_coef = lasso_test.named_steps['lasso'].coef_
    n_zero = np.sum(np.abs(lasso_coef) < 1e-10)
    n_features = len(lasso_coef)

    print(f"Alpha={alpha:6.3f}: {n_zero}/{n_features} zero coefficients")

    # Show which features are eliminated
    if n_zero > 0:
        zero_features = [X.columns[i] for i, coef in enumerate(lasso_coef) if abs(coef) < 1e-10]
        print(f"       Eliminated: {', '.join(zero_features) if zero_features else 'None'}")


LASSO REGRESSION CV RESULTS
CV RMSE scores: [np.float64(5.2264159959603855), np.float64(4.557090286618102), np.float64(5.519709704055303), np.float64(4.181760964645452), np.float64(4.780875252536945)]
CV RMSE - Mean: 4.8532, Std: 0.4748

Lasso Regression (Test Set)
  MAE : 4.321
  RMSE: 5.384
  R²  : 0.892

LASSO FEATURE SELECTION ANALYSIS
Alpha= 0.001: 0/5 zero coefficients
Alpha= 0.010: 0/5 zero coefficients
Alpha= 0.100: 0/5 zero coefficients
Alpha= 1.000: 0/5 zero coefficients
Alpha=10.000: 4/5 zero coefficients
       Eliminated: data_volume_mb, unusual_time, patch_age_days, admin_login


## Task 5 — Ridge vs Lasso Coefficients

**Goal:**  
Understand how different regularization methods affect model parameters.

**What to do:**  
- Inspect the learned coefficients from both models.

**What to observe:**  
- Which features are shrunk most?
- Does Lasso eliminate any features?




In [ ]:
# Student code here

# Extract coefficients from both models
ridge_coefs = ridge_pipe.named_steps['ridge'].coef_
lasso_coefs = lasso_pipe.named_steps['lasso'].coef_

# Create comparison DataFrame
coef_comparison = pd.DataFrame({
    'Feature': X.columns,
    'Ridge Coefficient': ridge_coefs,
    'Lasso Coefficient': lasso_coefs,
    '|Ridge|': np.abs(ridge_coefs),
    '|Lasso|': np.abs(lasso_coefs),
    'Difference': ridge_coefs - lasso_coefs
})

print("\n" + "="*60)
print("RIDGE VS LASSO COEFFICIENT COMPARISON")
print("="*60)
print(coef_comparison.to_string(index=False))

# Calculate summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Ridge - Mean |coef|: {np.abs(ridge_coefs).mean():.4f}")
print(f"Lasso - Mean |coef|: {np.abs(lasso_coefs).mean():.4f}")
print(f"Ridge - Std of coefs: {ridge_coefs.std():.4f}")
print(f"Lasso - Std of coefs: {lasso_coefs.std():.4f}")

# Identify most important features for each model
print("\n" + "="*60)
print("TOP FEATURES BY MAGNITUDE")
print("="*60)
print("\nRidge Top Features:")
ridge_top = coef_comparison.nlargest(3, '|Ridge|')[['Feature', 'Ridge Coefficient']]
print(ridge_top.to_string(index=False))

print("\nLasso Top Features:")
lasso_top = coef_comparison.nlargest(3, '|Lasso|')[['Feature', 'Lasso Coefficient']]
print(lasso_top.to_string(index=False))


RIDGE VS LASSO COEFFICIENT COMPARISON
       Feature  Ridge Coefficient  Lasso Coefficient   |Ridge|   |Lasso|  Difference
 failed_logins          10.325822          10.260206 10.325822 10.260206    0.065616
data_volume_mb           4.867032           4.779751  4.867032  4.779751    0.087281
  unusual_time           5.939430           5.859467  5.939430  5.859467    0.079963
patch_age_days           2.917478           2.831696  2.917478  2.831696    0.085783
   admin_login           7.677389           7.588953  7.677389  7.588953    0.088437

SUMMARY STATISTICS
Ridge - Mean |coef|: 6.3454
Lasso - Mean |coef|: 6.2640
Ridge - Std of coefs: 2.5187
Lasso - Std of coefs: 2.5248

TOP FEATURES BY MAGNITUDE

Ridge Top Features:
      Feature  Ridge Coefficient
failed_logins          10.325822
  admin_login           7.677389
 unusual_time           5.939430

Lasso Top Features:
      Feature  Lasso Coefficient
failed_logins          10.260206
  admin_login           7.588953
 unusual_time    

## Task 6 — Convert Risk Score to Binary Labels

**Goal:**  
Transform a regression problem into a binary classification problem.

**What to do:**  
- Label sessions as High Risk (1) or Low Risk (0) using a threshold of 70.

**What to observe:**  
- The task changes from predicting a continuous value to predicting a class label..


In [ ]:
# Student code here

THRESHOLD = 70

# Create binary labels
y_train_binary = (y_train >= THRESHOLD).astype(int)
y_test_binary = (y_test >= THRESHOLD).astype(int)

print("\n" + "="*50)
print("BINARY CLASSIFICATION SETUP")
print("="*50)
print(f"Threshold for High Risk: ≥ {THRESHOLD}")

# Class distribution
train_high_ratio = y_train_binary.mean()
test_high_ratio = y_test_binary.mean()

print(f"\nTraining Set:")
print(f"  High Risk (1): {y_train_binary.sum()} samples ({train_high_ratio:.1%})")
print(f"  Low Risk (0): {len(y_train_binary) - y_train_binary.sum()} samples ({1-train_high_ratio:.1%})")

print(f"\nTest Set:")
print(f"  High Risk (1): {y_test_binary.sum()} samples ({test_high_ratio:.1%})")
print(f"  Low Risk (0): {len(y_test_binary) - y_test_binary.sum()} samples ({1-test_high_ratio:.1%})")

# Verify conversion
print("\nSample conversions (first 10 test instances):")
sample_df = pd.DataFrame({
    'Original Risk': y_test.head(10).values,
    'Binary Label': y_test_binary.head(10).values
})
print(sample_df.to_string(index=False))


BINARY CLASSIFICATION SETUP
Threshold for High Risk: ≥ 70

Training Set:
  High Risk (1): 14 samples (3.5%)
  Low Risk (0): 386 samples (96.5%)

Test Set:
  High Risk (1): 2 samples (2.0%)
  Low Risk (0): 98 samples (98.0%)

Sample conversions (first 10 test instances):
 Original Risk  Binary Label
     52.605686             0
     47.723299             0
     31.538973             0
     16.944240             0
      4.301410             0
     57.327732             0
     25.452335             0
     57.654721             0
      4.808049             0
     40.715120             0


## Task 7 — Logistic Regression for Cyber Risk Classification

**Goal:**  
Classify network sessions as High Risk or Low Risk.

**What to do:**  
- Train a Logistic Regression model.
- Evaluate using confusion matrix and classification metrics.

**What to observe:**  
- Focus on recall and precision.
- Consider which metric is most important for cyber risk.


In [ ]:
# Student code here

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# Create pipeline with scaling and logistic regression
logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
)

# Train the model
logreg_pipe.fit(X_train, y_train_binary)

# Make predictions
y_pred_logreg = logreg_pipe.predict(X_test)
y_pred_proba = logreg_pipe.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("LOGISTIC REGRESSION RESULTS")
print("="*60)

# Calculate metrics
accuracy = accuracy_score(y_test_binary, y_pred_logreg)
precision = precision_score(y_test_binary, y_pred_logreg)
recall = recall_score(y_test_binary, y_pred_logreg)
f1 = f1_score(y_test_binary, y_pred_logreg)
conf_matrix = confusion_matrix(y_test_binary, y_pred_logreg)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nConfusion Matrix:")
print("              Predicted")
print("              Low    High")
print(f"Actual Low    {conf_matrix[0,0]:4d}   {conf_matrix[0,1]:4d}")
print(f"       High   {conf_matrix[1,0]:4d}   {conf_matrix[1,1]:4d}")

# Detailed classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test_binary, y_pred_logreg,
                          target_names=['Low Risk', 'High Risk']))

# Feature importance for classification
logreg_coefs = logreg_pipe.named_steps['logisticregression'].coef_[0]
coef_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': logreg_coefs,
    'Odds Ratio': np.exp(logreg_coefs),
    '|Coefficient|': np.abs(logreg_coefs)
}).sort_values('|Coefficient|', ascending=False)

print("\n" + "="*60)
print("FEATURE IMPORTANCE (LOGISTIC REGRESSION)")
print("="*60)
print("(Odds Ratio > 1 increases risk, < 1 decreases risk)")
print(coef_importance[['Feature', 'Coefficient', 'Odds Ratio']].to_string(index=False))


LOGISTIC REGRESSION RESULTS
Accuracy : 0.9800
Precision: 0.5000
Recall   : 1.0000
F1-Score : 0.6667

Confusion Matrix:
              Predicted
              Low    High
Actual Low      96      2
       High      0      2

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Low Risk       1.00      0.98      0.99        98
   High Risk       0.50      1.00      0.67         2

    accuracy                           0.98       100
   macro avg       0.75      0.99      0.83       100
weighted avg       0.99      0.98      0.98       100


FEATURE IMPORTANCE (LOGISTIC REGRESSION)
(Odds Ratio > 1 increases risk, < 1 decreases risk)
       Feature  Coefficient  Odds Ratio
 failed_logins     3.431908   30.935624
   admin_login     2.593297   13.373787
  unusual_time     1.473308    4.363644
data_volume_mb     1.363954    3.911631
patch_age_days     0.988772    2.687932


**Reflection Questions (Answer Below)**
1. Which regression model performed best and why?
2. How did Ridge and Lasso differ in their coefficients?
3. In cyber risk assessment, why is recall often more important than accuracy?



In [ ]:
print("""
1. According the results:
   • Linear Regression:  RMSE=5.388, R²=0.892
   • Polynomial:        RMSE=5.613, R²=0.883 (worse - overfitted)
   • Ridge (α=1.0):     RMSE=5.386, R²=0.892
   • Lasso (α=0.1):     RMSE=5.384, R²=0.892

   Ridge Regression performed slightly better than the others.
   It has the same R² score as linear regression but a tiny bit
   lower RMSE (5.386 vs 5.388). The polynomial model actually
   did worse because it tried too hard to fit the training data
   and couldn't generalize well to new data.

2.Ridge coefficients are about 0.08 larger than Lasso coefficients.
   Ridge shrinks all features but keeps them, while Lasso shrinks more
   and can remove features entirely (at alpha=10, Lasso removed 4/5 features).
3. Because missing a real attack (false negative) can cause huge damage,
   while investigating false alarms just wastes a little time. Better to
   check 100 safe sessions than miss one real attack.
  """)


1. According the results:
   • Linear Regression:  RMSE=5.388, R²=0.892
   • Polynomial:        RMSE=5.613, R²=0.883 (worse - overfitted)
   • Ridge (α=1.0):     RMSE=5.386, R²=0.892
   • Lasso (α=0.1):     RMSE=5.384, R²=0.892

   Ridge Regression performed slightly better than the others.
   It has the same R² score as linear regression but a tiny bit
   lower RMSE (5.386 vs 5.388). The polynomial model actually
   did worse because it tried too hard to fit the training data
   and couldn't generalize well to new data.

2.Ridge coefficients are about 0.08 larger than Lasso coefficients.
   Ridge shrinks all features but keeps them, while Lasso shrinks more
   and can remove features entirely (at alpha=10, Lasso removed 4/5 features).
3. Because missing a real attack (false negative) can cause huge damage,
   while investigating false alarms just wastes a little time. Better to
   check 100 safe sessions than miss one real attack.
  


## Final Checklist

- [x] All tasks completed
- [x] All cells run without errors
- [x] Outputs are visible
- [x] Renamed to `Lab1_Filmon_Mehari.ipynb`
- [x] Downloaded and submitted the `.ipynb` via the Canvas

